# Algoritmi Genetici

## Problema și reprezentarea ei

Problema pe care am propus ca GA-ul să o rezolve este de planificare a claselor.

Problema este pusă altfel: dându-se un număr de camere, de profesori, de grupe și de clase, să se planifice cât mai optim orarul profesorilor și grupelor
în baza cunoașterii claselor care trebuie să aibă loc.

În cadrul acestei probleme, pentru a modela domeniul, am definit următoarele entități:

- `Time`
    - folosit pentru a realiza planificarea (pentru a compara ore, intervale de timp și altele)
- `Professor`
    - folosit pentru a reprezenta un profesor
- `Room`
    - reprezintă o sală de curs
     - prezintă o mărime maximă
- `Group`
    - reprezintă o grupă, menită să susțină cursuri
- `Specialization`
    - caracterizează o specializare
- `Subject`
    - reprezintă un subiect de predat
- `Section`
    - reprezintă un curs, unde e precizat cine predă (profesorul), cui predă (grupa/grupele) și cât durează cursul
- `SectionSchedule`
    - reprezintă o soluție de planificare a orarului (gruparea de secții asociate în căror camere au loc și în ce intervale orare)

De asemenea, am considerat și următoarele:

- am considerat că fiecare profesor poate lucra în intervalul orar `start`, `end`, în oricare zi (interval fix, pentru a simplifica problema)
    - ca mențiune, fiecare profesor are propriul său orar (adică, spre exemplu, un profesor poate fi de la 13:00 la 19:30 în toate zilele,
        iar altul de la 08:00 la 16:20)
- s-a considerat că fiecare grupă poate veni la cursuri în timpul unei zi de lucru
- cursurile se țin între `start` și `end` (spre exemplu, încep la ora 08:00 și se termină la 21:10)
- prin curs ne referi la următoarele:
    - oră de predare de teorie (curs)
    - oră de practică (laborator)
    - oră de realizat exerciții (seminar)


Ca observați în privința implementării:

- Generarea unui individ poate prezenta probleme, întrucât nu se verifică dacă un profesor sau grupă au 2 sau mai mult clase în același interval orar.
Camerele pot fi folosite în același timp pentru cursuri (evident, această soluție este invalidă, dar penalizarea unei configurați invalid
ar trebui să elimine în timp astfel de probleme)
- Generarea unei planificări pentru curs (unei gene), ține cont de mărimea grupei și a camerei

## Funcția obiectiv


Funcția de obiectiv a fost realizată astfel:

- Penalizarea mare a configurațiilor invalide (`Hard constraints`):
    - dacă un profesor ajunge să predea 2 sau mai multe cursuri în același timp, se aplică o penalizare de 100000
    - dacă o grupă are 2 sau mai multe cursuri simultan, se aplică o penalizare de 100000
    - dacă o cameră are 2 sau mai multe cursuri ținute simultan, se aplică la fel o penalizare de 100000


- Penalizarea redusă a configurațiilor care nu sunt optime (`Soft constraints`):
    - se penalizează timpul în care profesori stau în cadrul facultății nepredând ore (gap-uri între ore)
        -  se înmulțește cu 4 (pentru a-i spori importanța)
    - asemănător, se penalizează la fel și timpul în care grupele stau în cadrul facultății fără ore
        - se înmulțește cu 3 (puțin mai important comparativ cu timpul mort al profesorilor)
    - se penalizează planificarea unui curs cu un număr redus de studenți într-o sală mai mare
        - se înmulțește cu 2 (considerând că tot e mai puțin important decât cele 2 anterioare, întrucât nu afectează cu așa mult că orele au loc în săli mai mari,
                deși nu e preferabil)

- Într-un final, acestea se adună (pentru a avea penalizarea totală)

- Se întoarce ca valoare `1 / (1 + penalizare)`

Se observă că problema este de maximizare (cu cât funcția obiectiv e mai mare, cu atât mai bună soluția).

Funcția obiectiv a fost aleasă ca `1 / (1 + penalizare)` întrucât ajută să avem mai multe detalii în legătură cu soluția găsită și cât de bună este:
   - dacă are valoare `1`, atunci soluția este cea mai bună (întrucât am avea că `penalizare=0`)

   - dacă are valoare `0` (evident, din motive de convergență a funcției nu va fi, dar se va apropia), atunci soluția este foarte rea (întrucât `penalizare`
    are o valoare foarte mare, raportul tinzând spre 0




## Codificarea soluției

Codificarea soluției s-a realizat considerând că un cromozom (o soluție), este format dintr-un vector de `SectionSchedule`-uri (unde un obiect de
acest tip reprezintă un curs care trebuie să aibă loc, sala aleasă pentru acesta și grupa/grupele și profesorul care predă cursul)

S-a considerat următoarul lucru în cadrul acestei codificări:

- dacă pe poziția `k` din vectorul de `SectionSchedule` a fost amplasat un anumit curs (`Section`), atunci acel curs va rămâne acolo, tot ce se modifică fiind
camera alocată pentru a-l ține

De asemenea, mai precizăm și următoarele observații:

- codificarea, cu toate că e simplă, conduce la probleme în cadrul calculelor funcției obiectiv pentru fiecare soluție, întrucât trebuie analizate
restul soluțiilor cu cea curentă pentru a atribui un penalty (ceea ce pentru `n` cursuri implică o parcurgere de `O(n^2)` din punct
de veder al timpului de execuție, ceea ce nu este ideal, dar s-a considerat că numărul de cursuri nu ar trebuie să fie într-un final așa mare)


## Metoda de selecție

Pentru selecția celor 2 părinți, am folosit următorul algoritm de tip `Tournament Selection`

- Am selectat `5` indivizi la întâmplare
- L-am ales pe cel cu cel mai mare scor de fitness ca fiind primul părinte
- Același lucru am făcut iar, pentru a selecta cel de al 2 lea părinte

Abordarea aceasta, deși simplistă, combină două aspecte:
    - selectarea la întâmplare a 2 părinții (pentru a avea diversitate în cadrul operației de cross-over)
    - exploatarea soluțiilor bune (prin selectarea celor mai bune soluții alese la întâmplare)

Am ales această metodă mai mult pentru a avea parte de diversitate în cadrul alegeri părinților (prin selecția la întâmplare a unui număr
destul de mic din populație), dar și pentru a spori transmiterea de gene bune pentru viitoarele generații (prin alegerea celor mai bune dintre
soluțiile alese la întâmplare)


## Metoda de crossover



Pentru cross-over, m-am folosit de metoda de tip `One Point`, astfel:

- Pe baza lungimii unui cromozom (întrucât cromozomi sunt de aceași lungime, fiecare având de planificat toate cele `n` cursuri, având lungimea `n`),
se alege aleator un `k` între `1` și `n-1`

- Acel punct va fi folosit ca punct de tăiere (întrucât știm că fiecare genă reprezintă un anumit curs planificat, iar acel curs nu se mută)
- Pentru a respecta ideea de a nu muta cursuri în cadrul cromozomului, fiii vor fi obținuți astfel:

````
# exemplu pentru k = 2

Parent1 = [gp1, gp2, gp3, ...., gpn]
Parent2 = [ep1, ep2, ep3, ...., epn]

# Se face tăiere

# de la al k-lea element a început tăierea
# se combină cei 2 părinți, genele unuia combinându-se cu ale altuia, dar neschimbându-și semnificația (același curs rămâne la aceea poziție)

Child1 = [gp1, gp2, ep3, ...., epn]
Child2 = [ep1, ep2, gp3, ...., gpn]
````

Această metodă a fost aleasă întrucât este simplă, iar de asemenea nu distruge soluțiile (întrucât pozițiile genelor semnifică ceva, schimbarea lor
ducând la soluții inexplicabile, cum ar fi planificarea unui curs de 4 ori pe săptămână spre exemplu)

De asemenea, alegerea punctului de tăiere la întâmplare poate ajuta și în cazul în care dorim să sporim diversitatea, nefiind mereu combinate aceleași gene



## Metoda de mutație

În cadrul metode de mutație, am ales să interschimb la întâmplare camera aleasă (care poate fi posibil aleasă) și
ziua în care are loc cursul

Am ales această metodă în speranța de a alege poate o cameră mai mică (reducând astfel penalizarea în cadrul mărimii camerei în raport
cu numărul de studenți), iar de asemenea schimbarea zilei poate conduce la o soluție care să mute un curs indirect într-o porțiune
de ore consecutive care să prezinte cursuri (astfel, reducând penalizarea)


- Rata de mutație a fost aleasă de a fi 10%, pe raționamentul că, deși este bine să avem diversitate, tot ar trebui să evită să avem soluții extrem
de întâmplătoare (datorită mutației), întrucât nu am converge spre o soluție optimă. Evident, pentru a optimiza găsirea unei soluții optime, am mizat
și pe un crossover rate de 85%, pentru a promova diversitatea datorată combinării dintre părinți.


## Elitism

S-a folosit elitism în cadrul implementării astfel:

- la fiecare generație, cei mai buni `4` indivizi au fost trecuți automat în generația următoare
- aceștia sunt considerați ca parte a populației în cadrul procesului de selecție

Am ales să folosesc elitismul întrucât am dorit să nu se piardă soluțiile cele mai bune între generații. Evident, pentru a evita
o convergență prematură, numărul redus de indivizii (4) care sunt trecuți automat în următoarea etapă și selecția de tip turneu
cu 5 indivizii (care scade semnificativ șansele ca un individ trecut automat să fie selectat), conduc la evitarea acestui fenomen,
păstrând diversitate (prin selecție aleatoare), dar și ținând cont de cei mai buni indivizi, ținând cont că s-ar putea să ajute în găsirea unor
soluții mai bune.




# Aplicația

S-a realizat pentru exemplu, folosind date de la `https://orar.ulbsibiu.ro/inginerie/`, orarul pentru grupele de Calculatoare, TI și ISM an 4
(pentru ca utilizarea să fie cât mai aproape de realitate), dar anonimizând numele profesorilor



In [ ]:
# Pachetele necesare

!git clone https://github.com/radumusoaie20/intelligent-systems.git
%cd intelligent-systems

In [ ]:
# Avem nevoie de dependențele necesare

!pip install rich

Se va prezenta cum se poate apela programul principal pentru a obține rezultate. Codul sursă se află în secțiune de `Cod Sursă`



In [ ]:
from genetic_algorithm.impl.genetic_algorithm import GeneticAlgorithm
from genetic_algorithm.work.class_scheduling.class_scheduling import (make_section_slots, make_create_individual, select_func, crossover_func,
                                                                      make_mutation, make_fitness_func)
from genetic_algorithm.work.class_scheduling.domain import *
from genetic_algorithm.work.class_scheduling.print_utils import interactive_schedule

# Time

t_08_00 = Time(8, 0, 0)
t_09_40 = Time(9, 40, 0)
t_11_20 = Time(11, 20, 0)
t_13_00 = Time(13, 0, 0)
t_14_40 = Time(14, 40, 0)
t_16_20 = Time(16, 20, 0)
t_18_00 = Time(18, 0, 0)
t_19_40 = Time(19, 40, 0)
t_21_10 = Time(21, 10, 0)

day_start = Time(8, 0, 0)
day_end = Time(21, 10, 0)
pause_time = Time(0, 10, 0)

duration = Time(1, 30, 0)

# Specializari

spec_calc = Specialization('Computer Engineering')
spec_ti = Specialization('Information Technology')
spec_ism = Specialization('Multimedia System Engineering')

specializations = [spec_calc, spec_ti, spec_ism, spec_ism]


# Grupe

g_c_41_1 = Group('C_41/1', spec_calc, 17)
g_c_41_2 = Group('C_41/2', spec_calc, 16)
g_c_42_1 = Group('C_42/1', spec_calc, 13)
g_c_42_2 = Group('C_42/2', spec_calc, 15)
g_c_42_3 = Group('C_42/3', spec_calc, 12)

g_ti_41 = Group('TI_41', spec_ti, 18)

g_ism_41_1 = Group('ISM_41/1', spec_ism, 13)
g_ism_41_2 = Group('ISM_41/2', spec_ism, 11)

groups = [g_c_41_1, g_c_41_2, g_c_42_1, g_c_42_2, g_c_42_3,
          g_ti_41, g_ism_41_1, g_ism_41_2]

# Profesori

prof_ga = Professor('GA', t_09_40, t_21_10)
prof_md = Professor('MD', t_09_40, t_21_10)
prof_cv = Professor('CV', t_16_20, t_18_00)
prof_zb = Professor('ZB', t_18_00, t_21_10)
prof_pa = Professor('PA', t_16_20, t_21_10)
prof_fa = Professor('FA', t_08_00, t_21_10)
prof_br = Professor('BR', t_09_40, t_19_40)
prof_bm = Professor('BM', t_16_20, t_21_10)
prof_cd = Professor('CD', t_14_40, t_21_10)
prof_nm = Professor('NM', t_08_00, t_21_10)
prof_ma = Professor('MA', t_11_20, t_21_10)
prof_pv = Professor('PV', t_08_00, t_21_10)
prof_bs = Professor('BS', t_08_00, t_19_40)
prof_bi = Professor('BI', t_16_20, t_19_40)
prof_zd = Professor('ZD', t_18_00, t_21_10)
prof_nc = Professor('NC', t_08_00, t_21_10)
prof_pi = Professor('PI', t_08_00, t_21_10)
prof_cc = Professor('CC', t_08_00, t_19_40)
prof_ba = Professor('BA', t_18_00, t_21_10)
prof_sa = Professor('SA', t_18_00, t_21_10)
prof_cr = Professor('CR', t_08_00, t_14_40)
prof_bre =  Professor('BRE', t_09_40, t_19_40)

professors = [
    prof_ga, prof_md, prof_cv, prof_zb,
    prof_pa, prof_fa, prof_br, prof_cd,
    prof_nm, prof_ma, prof_pv,
    prof_bi, prof_bs, prof_zd, prof_nc, prof_pi,
    prof_cc, prof_sa, prof_cr, prof_bre,
    prof_bm, prof_ba
]

# Subiecte

sub_img_proc = Subject('Prelucrarea imaginilor')
sub_ml = Subject('Invatare automata')
sub_android = Subject('Elemente de informatica mobila')
sub_int_sys = Subject('Sisteme inteligente')
sub_soac = Subject('Simularea si optimizarea arhitecturilor de calcul')
sub_cybersec = Subject('Securitatea datelor')
sub_signal = Subject('Procesarea semnalelor')
sub_game_prg = Subject('Programarea jocurilor')
sub_encoding = Subject('Codificarea informatiei multimedia')
sub_discrete_sys = Subject('Sisteme dinamice cu evenimente discrete')

subjects = [
    sub_img_proc, sub_ml, sub_android, sub_int_sys,
    sub_soac, sub_cybersec, sub_signal, sub_game_prg,
    sub_encoding, sub_discrete_sys
]


# Sali

r_im_414 = Room('IM414', 23)
r_ie_305 = Room('IE305', 17)
r_im_201 = Room('IM201', 120)
r_im_216 = Room('IM216', 19)
r_im_405 = Room('IM405', 120)
r_ie_006 = Room('IE006', 30)
r_ie_113 = Room('IE113', 20)
r_im_321 = Room('IM321', 23)
r_ie_002 = Room('IE002', 22)
r_im_219 = Room('IM219', 25)
r_im_323 = Room('IM323', 40)
r_ie_303 = Room('IE303', 40)
r_ie_003 = Room('IE003', 30)
r_ie_304 = Room('IE304', 24)
r_ntt_data = Room('NTTData_Evolution', 27)
r_ie_101 = Room('IE101', 40)
r_im_320 = Room('IM320', 23)

rooms = [
    r_im_414, r_ie_305, r_im_201, r_im_216, r_im_405, r_ie_006, r_ie_113,
    r_im_321, r_ie_002, r_im_219, r_im_323, r_ie_303, r_ie_003,
    r_ie_304, r_ntt_data, r_ie_101, r_im_320
]

# Cursuri

g_c = {g_c_41_1, g_c_41_2, g_c_42_1, g_c_42_2, g_c_42_3}
g_c_ti = {g_c_41_1, g_c_41_2, g_c_42_1, g_c_42_2, g_c_42_3, g_ti_41}

sections = [
    # ISM

    # Courses
    Section(MeetingType.COURSE, sub_game_prg, {g_ism_41_1, g_ism_41_2}, prof_pi, duration),
    Section(MeetingType.COURSE, sub_ml, {g_ism_41_1, g_ism_41_2}, prof_ma, duration),
    Section(MeetingType.COURSE, sub_encoding, {g_ism_41_1, g_ism_41_2}, prof_bre, duration),
    Section(MeetingType.COURSE, sub_signal, {g_ism_41_1, g_ism_41_2}, prof_nc, duration),
    Section(MeetingType.COURSE, sub_img_proc, {g_ism_41_1, g_ism_41_2}, prof_br, duration),
    Section(MeetingType.COURSE, sub_discrete_sys, {g_ism_41_1, g_ism_41_2}, prof_cr, duration),
    Section(MeetingType.COURSE, sub_android, {g_ism_41_1, g_ism_41_2}, prof_md, duration),
    # Lab
    Section(MeetingType.LAB, sub_game_prg, {g_ism_41_1, g_ism_41_2}, prof_sa, duration),
    Section(MeetingType.LAB, sub_ml, {g_ism_41_1}, prof_cc, duration),
    Section(MeetingType.LAB, sub_ml, {g_ism_41_2}, prof_cc, duration),
    Section(MeetingType.LAB, sub_encoding, {g_ism_41_1}, prof_ba, duration),
    Section(MeetingType.LAB, sub_encoding, {g_ism_41_2}, prof_ba, duration),
    Section(MeetingType.LAB, sub_signal, {g_ism_41_1}, prof_nc, duration),
    Section(MeetingType.LAB, sub_signal, {g_ism_41_2}, prof_nc, duration),
    Section(MeetingType.LAB, sub_img_proc, {g_ism_41_1}, prof_nc, duration),
    Section(MeetingType.LAB, sub_img_proc, {g_ism_41_2}, prof_nc, duration),
    Section(MeetingType.LAB, sub_android, {g_ism_41_1}, prof_md, duration),
    Section(MeetingType.LAB, sub_android, {g_ism_41_2}, prof_md, duration),
    Section(MeetingType.LAB, sub_discrete_sys, {g_ism_41_1}, prof_cr, duration),
    Section(MeetingType.LAB, sub_discrete_sys, {g_ism_41_2}, prof_cr, duration),

    # TI

    # Courses
    Section(MeetingType.COURSE, sub_android, g_c_ti, prof_cv, duration),
    Section(MeetingType.COURSE, sub_int_sys, g_c_ti, prof_zb, duration),
    Section(MeetingType.COURSE, sub_signal, {g_ti_41}, prof_nc, duration),
    Section(MeetingType.COURSE, sub_img_proc, {g_ti_41}, prof_br, duration),
    Section(MeetingType.COURSE, sub_soac, g_c_ti, prof_fa, duration),
    Section(MeetingType.COURSE, sub_cybersec, g_c_ti, prof_br, duration),
    Section(MeetingType.COURSE, sub_ml, g_c_ti, prof_md, duration),
    # Lab
    Section(MeetingType.LAB, sub_android, {g_ti_41}, prof_md, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_ti_41}, prof_ma, duration),
    Section(MeetingType.LAB, sub_signal, {g_ti_41}, prof_nc, duration),
    Section(MeetingType.LAB, sub_img_proc, {g_ti_41}, prof_ga, duration),
    Section(MeetingType.LAB, sub_soac, {g_ti_41}, prof_pa, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_ti_41}, prof_pv, duration),
    Section(MeetingType.LAB, sub_ml, {g_ti_41}, prof_bs, duration),

    # C

    # Courses
    Section(MeetingType.COURSE, sub_signal, g_c, prof_nm, duration),
    Section(MeetingType.COURSE, sub_img_proc, g_c, prof_br, duration),

    # Lab

    Section(MeetingType.LAB, sub_img_proc, {g_c_41_1}, prof_ga, duration),
    Section(MeetingType.LAB, sub_ml, {g_c_41_1}, prof_md, duration),
    Section(MeetingType.LAB, sub_soac, {g_c_41_1}, prof_nc, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_c_41_1}, prof_br, duration),
    Section(MeetingType.LAB, sub_android, {g_c_41_1}, prof_md, duration),
    Section(MeetingType.LAB, sub_signal, {g_c_41_1}, prof_nm, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_c_41_1}, prof_ma, duration),

    Section(MeetingType.LAB, sub_img_proc, {g_c_41_2}, prof_ga, duration),
    Section(MeetingType.LAB, sub_ml, {g_c_41_2}, prof_md, duration),
    Section(MeetingType.LAB, sub_soac, {g_c_41_2}, prof_pa, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_c_41_2}, prof_bm, duration),
    Section(MeetingType.LAB, sub_android, {g_c_41_2}, prof_md, duration),
    Section(MeetingType.LAB, sub_signal, {g_c_41_2}, prof_nm, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_c_41_2}, prof_ma, duration),

    Section(MeetingType.LAB, sub_img_proc, {g_c_42_1}, prof_ga, duration),
    Section(MeetingType.LAB, sub_ml, {g_c_42_1}, prof_bs, duration),
    Section(MeetingType.LAB, sub_soac, {g_c_42_1}, prof_bi, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_c_42_1}, prof_pv, duration),
    Section(MeetingType.LAB, sub_android, {g_c_42_1}, prof_md, duration),
    Section(MeetingType.LAB, sub_signal, {g_c_42_1}, prof_zb, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_c_42_1}, prof_ma, duration),

    Section(MeetingType.LAB, sub_img_proc, {g_c_42_2}, prof_ga, duration),
    Section(MeetingType.LAB, sub_ml, {g_c_42_2}, prof_bs, duration),
    Section(MeetingType.LAB, sub_soac, {g_c_42_2}, prof_bi, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_c_42_2}, prof_pv, duration),
    Section(MeetingType.LAB, sub_android, {g_c_42_2}, prof_md, duration),
    Section(MeetingType.LAB, sub_signal, {g_c_42_2}, prof_zb, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_c_42_2}, prof_ma, duration),

    Section(MeetingType.LAB, sub_img_proc, {g_c_42_3}, prof_ga, duration),
    Section(MeetingType.LAB, sub_ml, {g_c_42_3}, prof_bs, duration),
    Section(MeetingType.LAB, sub_soac, {g_c_42_3}, prof_pa, duration),
    Section(MeetingType.LAB, sub_cybersec, {g_c_42_3}, prof_bm, duration),
    Section(MeetingType.LAB, sub_android, {g_c_42_3}, prof_cd, duration),
    Section(MeetingType.LAB, sub_signal, {g_c_42_3}, prof_nm, duration),
    Section(MeetingType.LAB, sub_int_sys, {g_c_42_3}, prof_ma, duration),
]

# precompute time slots
section_slots = make_section_slots(sections, day_start, day_end, pause_time)

# need a creator for individual chromosomes
create_individual = make_create_individual(sections, section_slots, rooms)

# mutation function
mutation_func = make_mutation(section_slots, rooms)

# fitness function
fitness_func = make_fitness_func(day_start, day_end)

# GA
ga = GeneticAlgorithm(
    population_size=50,
    fitness_func=fitness_func,
    create_individual_func=create_individual,
    selection_func=select_func,
    crossover_func=crossover_func,
    mutation_func=mutation_func,
    crossover_rate=0.85,
    mutation_rate=0.1,
    elitism_count=4,
    minimize_solution=False
)

best, best_f = ga.run(num_generations=50, verbose=True)

print(f"Gasit configuratia cu scor: {best_f}")

# Interactive tabular views
interactive_schedule(best, day_start, day_end, duration, pause_time, groups, professors, rooms)

Prin `interactive_schedule` putem vizualiza pentru diferiți profesori, grupe sau săli cum au fost planificate pentru cele 5 zile.



# Cod Sursă

## domain.py

````python
from enum import Enum

class Time:
    def __init__(self, hour, minute, second):
        self.hour = hour
        self.minute = minute
        self.second = second

    def __copy__(self):
        return Time(self.hour, self.minute, self.second)

    def to_seconds(self):
        return (self.hour * 3600) + (self.minute * 60) + self.second

    def __eq__(self, other):
        return self.hour == other.hour and self.minute == other.minute and self.second == other.second

    def __lt__(self, other):
        this_seconds = self.to_seconds()
        other_seconds = other.to_seconds()
        return this_seconds < other_seconds

    def __gt__(self, other):
        this_seconds = self.to_seconds()
        other_seconds = other.to_seconds()
        return this_seconds > other_seconds

    def __le__(self, other):
        this_seconds = self.to_seconds()
        other_seconds = other.to_seconds()
        return this_seconds <= other_seconds

    def __ge__(self, other):
        this_seconds = self.to_seconds()
        other_seconds = other.to_seconds()
        return this_seconds >= other_seconds

    def __int__(self):
        """
        :return: The seconds for this `Time` object
        """
        return self.hour * 60 * 60 + self.minute * 60 + self.second

    def __add__(self, other):
        """
        Adds this `Time` object to `other`
        :param other: The other `Time` object
        :return: A `Time` object representing the sum of this `Time` object and `other`, wrapping around a day (24 hours)
        """
        total_seconds = int(self) + int(other)
        day_seconds = 24 * 60 * 60
        return time_from_seconds(total_seconds % day_seconds)

    def __sub__(self, other):
        """
        Adds this `Time` object to `other`
        :param other: The other `Time` object
        :return: A `Time` object representing the difference of this `Time` object and `other`, wrapping around a day (24 hours)
        """
        total_seconds = int(self) - int(other)
        day_seconds = 24 * 60 * 60
        return time_from_seconds(total_seconds % day_seconds)

    def __str__(self):
        hour = "0" + str(self.hour) if len(str(self.hour)) == 1 else self.hour
        minute = "0" + str(self.minute) if len(str(self.minute)) == 1 else self.minute
        second = "0" + str(self.second) if len(str(self.second)) == 1 else self.second
        return f'{hour}:{minute}:{second}'

    def __hash__(self):
        return self.to_seconds()


def time_from_seconds(seconds: int):
    hour = seconds // 3600
    minute = seconds % 3600 // 60
    second = seconds % 60
    return Time(hour, minute, second)

def time_from_start_and_duration(start: Time, duration_in_minutes: int) -> (Time, Time):
    return start, time_from_seconds(int(start) + duration_in_minutes * 60)  # returns in seconds


class Professor:
    def __init__(self, name: str, start_hour: Time, end_hour: Time):
        """
        Constructs a professor. For example
        :param name: The name of the professor
        :param start_hour: The start hour of the professor (when he can teach)
        :param end_hour: The end hour of the professor (when he has to leave and cannot teach)
        """
        self.name = name
        self.start_hour = start_hour
        self.end_hour = end_hour

    def __eq__(self, other):
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Specialization:
    def __init__(self, name: str):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Room:
    def __init__(self, name: str, max_size: int):
        self.name = name
        self.max_size = max_size

    def __eq__(self, other):
        return self.name == other.name and self.max_size == other.max_size

    def __hash__(self):
        return hash(self.name) + 11 * self.max_size

class Group:
    def __init__(self, name: str, specialization: Specialization, size: int):
        self.specialization = specialization
        self.name = name
        self.size = size

    def __eq__(self, other):
        return self.specialization == other.specialization and self.name == other.name

    def __hash__(self):
        return hash(self.name) + 31 * self.size + hash(self.specialization) * 13

class MeetingType(Enum):
    COURSE = 1
    LAB = 2
    SEMINAR = 3

class Subject:
    def __init__(self, name: str):
        """
        Constructs a subject.
        :param name: The name of the subject
        """
        self.name = name

    def __eq__(self, other):
        return  self.name == other.name or self.name in other.__other_names

    def __hash__(self):
        return hash(self.name)

class Section:
    def __init__(self, meeting_type: MeetingType, subject: Subject, group: set[Group], professor: Professor,
                 duration: Time):
        """
        Constructs a section.
        :param meeting_type: The type of meeting (laboratory, course or seminar)
        :param subject: The subject of the meeting
        :param group: The group(s) that is having the meeting
        :param professor: The professor assigned to hold the meeting
        :param duration: The duration of the meeting
        """
        self.meeting_type = meeting_type
        self.subject = subject
        self.group = group
        self.professor = professor
        self.duration = duration

    def __eq__(self, other):
        return self.meeting_type == other.meeting_type and self.subject == other.subject and self.group == other.group and self.professor == other.professor and self.duration == other.duration

    def __hash__(self):
        return hash(self.meeting_type) * 13 + hash(frozenset(self.group)) * 31 + hash(self.professor) * 17 + hash(self.duration) * 23 + hash(self.subject) * 17

class SectionSchedule:
    def __init__(self, section: Section, room: Room, time_start: Time, day: int):
        """
        Constructs a section schedule.
        :param section: The section for which the schedule will be created
        :param time_start: The start time of the section
        :param room: The room of the section
        :param day: The day of the week
        """

        self.students_size = sum(g.size for g in section.group)
        self.section = section
        self.time_start = time_start
        self.time_end = time_start + section.duration
        self.room = room
        self.day = day

    def exceeds_room_size(self) -> bool:
        """
        Checks if the number of students exceeds the room size.
        :return: `True` if the number of students exceeds the room size, `False` otherwise
        """
        return self.students_size > self.room.max_size

    def is_outside_of_professor_time(self) -> bool:
        """
        Checks if the section is outside the professor time.
        :return: `True` if the section is outside the professor time, `False` otherwise
        """
        prof = self.section.professor
        return not (self.time_start >= prof.start_hour and self.time_end <= prof.end_hour)

    def is_before(self, other):
        """
        :param other: The other `SectionSchedule`
        :return: `True` if the section is before the other `SectionSchedule`
        """
        return self.time_end + self.day * 24 * 60 * 60 < other.time_start + other.day * 24 * 60 * 60

    def is_after(self, other):
        """
       :param other: The other `SectionSchedule`
       :return: `True` if the section is after the other `SectionSchedule`
       """
        return self.time_start + self.day * 24 * 60 * 60 > other.time_end + other.day * 24 * 60 * 60

    def __eq__(self, other):
        return (self.section == other.section and self.time_start == other.time_start and self.time_end == other.time_end
                    and self.room == other.room and self.day == other.day)

    def __str__(self):
        return (f"Subject: {self.section.subject.name} \n"
                f"Professor: {self.section.professor.name} \n"
                f"Room: {self.room.name}, Size: {self.room.max_size} \n"
                f"Number of students: {self.students_size} \n"
                f"Day: {self.day} \n"
                f"{str(self.time_start)} - {self.time_end} \n")

    def __hash__(self):
        return self.time_start.__hash__() * 17 + self.section.__hash__() * 13 + self.room.__hash__() * 31 + self.day * 23
````

## class_scheduling.py

````python
from random import choice, sample, randint
from copy import copy

from genetic_algorithm.work.class_scheduling.domain import *

# Chromosome length will be determined by the number of classes that need to be scheduled (each class needed to be taught represents one gene)

# Utility

def section_time_slots(section: Section, start: Time, end: Time, pause_time: Time) -> list[(Time, Time)]:

    result: list[(Time, Time)] = []

    # We will limit the `start` and `end` to the professors available time
    start = max(start, section.professor.start_hour) # Profs mostly start later (a bigger time value)
    end = min(end, section.professor.end_hour) # Profs cannot teach after their work day is over (which is less than the end time)

    current_state = copy(start)

    while current_state + section.duration <= end:
        result.append((copy(current_state), current_state + section.duration))
        current_state = current_state + section.duration + pause_time

    return result

def make_section_slots(sections, day_start, day_end, pause_time):
    section_slots = {}
    for sec in sections:
        # limit day window to professor availability
        prof_start = max(day_start, sec.professor.start_hour)
        prof_end = min(day_end, sec.professor.end_hour)
        # generate slots for that section's duration (returns list of (start, end) Time)
        slots = section_time_slots(sec, prof_start, prof_end, pause_time)
        if not slots:
            raise ValueError(f"No valid slots for section {sec} given professor/working hours")
        section_slots[sec] = slots
    return section_slots


# We will take the approach of generating valid scheduling objects (meaning that it respects the prof schedule and the room size)
# The fitness function will hard penalize all the resulting individuals (`SectionSchedule`) in case they
# have overlapping rooms or professors or groups
# We need a function with no params, so we need to use a wrapper to get the method

def make_create_individual(sections, section_slots, rooms):
    def create_individual_solution():
        chromosome = []
        for section in sections:

            if not len(section_slots[section]):
                raise Exception('No time available to take the section {} with the prof'.format(section, section.professor))

            start, _ = choice(section_slots[section])  # Pick a random time slot for the section

            possible_rooms = [r for r in rooms if r.max_size >= sum(g.size for g in section.group)]

            if not len(possible_rooms):
                raise Exception('No rooms available for section {}'.format(section))

            day = randint(0, 4)

            room = choice(possible_rooms)  # pick a random room from the possible rooms
            chromosome.append(SectionSchedule(section, room, start, day))

        return chromosome

    return create_individual_solution


# Parent selection (tournament selection)
def select_func(population: list[list[SectionSchedule]], fitness_scores: list[float]):

   candidates = sample(list(enumerate(population)), k=5)
   parent1, parent2 = None, None
   max1, max2 = 0, 0
   p1_idx = 0

   for i, section in candidates:
       if fitness_scores[i] > max1:
           max1 = fitness_scores[i]
           parent1 = section
           p1_idx = i

   candidates = sample(candidates, k=5)
   for i, section in candidates:
       if fitness_scores[i] > max2 and i != p1_idx:
           max2 = fitness_scores[i]
           parent2 = section

   return parent1, parent2


# Cross-over
# A solution is basically a list of schedules for sections (the order is maintained because sections is a list)
# We will do a single point crossover
def crossover_func(parent1: list[SectionSchedule], parent2: list[SectionSchedule]):

    size = len(parent1)
    point = randint(1, size - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2


# Mutation

def make_mutation(section_slots, rooms):
    def mutation_func(chromosome: list[SectionSchedule]):
        for i, gene in enumerate(chromosome):

            # pick time slot that fits professor day
            start, end = choice(section_slots[gene.section])

            # pick valid room
            possible_rooms = [r for r in rooms if r.max_size >= sum(g.size for g in gene.section.group)]
            room = choice(possible_rooms)

            # pick another day maybe?
            day = randint(0, 4)

            chromosome[i] = SectionSchedule(gene.section, room, start, day)

            return chromosome

    return mutation_func

# Fitness evaluation
# As a note, our solution doesn't filter out impossible solutions (like a professor having 2 courses at the same time)
# We are relying on the fact that since these solutions tend to bring a big penalty, they will cease to propagate throughout generations
# leading to their disappearance
# So this means that we are taking a risk in case the initial population tends to spawn invalid solutions

def make_fitness_func(day_start: Time, day_end: Time):
    def fitness_func(chromosome: list[SectionSchedule]) -> float:
        penalty = 0

        # Check overlaps for the chromosome (solution) schedules (for each pair)
        # Kind of like doing bubble sort

        # Hard constraints mostly (big penalty yields in almost 0 chance of selecting it as a viable solution)
        for i, a in enumerate(chromosome):
            for j, b in enumerate(chromosome):
                if i >= j: continue # mimics bubble sort, finding pairs

                # Room conflict (both sections take place in the same room at approximately the same time)
                if a.room == b.room and not (a.is_before(b) or a.is_after(b)):
                    penalty += 100000

                # Professor conflict (the sections are being taught by the same professor at approximately the same time)
                if a.section.professor == b.section.professor and not (a.is_before(b) or a.is_after(b)):
                    penalty += 100000

                # Group conflict (a group takes two sections at the same time)
                if not a.section.group.isdisjoint(b.section.group) and not (a.is_before(b) or a.is_after(b)):
                    penalty += 100000

        # Soft constraints (Help with optimization)

        # Minimize professor gaps
        penalty += professor_idle_penalty(chromosome, day_start, day_end) * 4

        # Minimize groups gaps
        penalty += group_idle_penalty(chromosome, day_start, day_end) * 3

        # Penalize placing a small number of students in big rooms
        penalty += room_size_penalty(chromosome) * 2

        # we want to maximize this
        # when penalty is 0, we have a value of 1, meaning a perfect solution (no penalties)
        # as it tends towards 0, we have worse solutions
        return 1 / (1 + penalty)


    return fitness_func

def professor_idle_penalty(chromosome: list[SectionSchedule], day_start: Time, day_end: Time) -> float:
    penalty = 0

    # Given the solution, we have to find out for the professor his schedule
    prof_sections = {}
    for schedule in chromosome:
        prof = schedule.section.professor
        prof_sections.setdefault(prof, []).append(schedule)


    for prof, sections in prof_sections.items():

        # Grouping by day
        sections_by_day = {}
        for section in sections:
            sections_by_day.setdefault(section.day, []).append(section)

        for day, day_sections in sections_by_day.items():

           # Gap before first class
           first = day_sections[0]
           penalty += int(first.time_start - first.section.professor.start_hour) // 60

           # Gap between consecutive classes
           day_sections.sort(key=lambda s: int(s.time_start)) # sorting within the day

           for i in range(len(day_sections) - 1):
            current_end = int(day_sections[i].time_end)
            next_start = int(day_sections[i + 1].time_start)

            gap_minutes = (next_start - current_end) // 60

            penalty += gap_minutes

            # Gap between last class and end
            last = day_sections[-1]
            penalty += int(last.section.professor.end_hour - last.time_end) // 60

    return penalty

def group_idle_penalty(chromosome: list[SectionSchedule], day_start: Time, day_end: Time) -> float:

    penalty = 0

    group_sections = {}
    for schedule in chromosome:
        for g in schedule.section.group:
            group_sections.setdefault(g, []).append(schedule)


    for g, sections in group_sections.items():

        # Group by day
        sections_by_day = {}
        for section in sections:
            sections_by_day.setdefault(section.day, []).append(section)

        for day, day_sections in sections_by_day.items():

            # Gap before first class
            first = day_sections[0]
            penalty += int(first.time_start - day_start) // 60

            day_sections.sort(key=lambda s: int(s.time_start))

            for i in range(len(day_sections) - 1):
                current_end = int(day_sections[i].time_end)
                next_start = int(sections[i + 1].time_start)
                gap_minutes = (next_start - current_end) // 60

                penalty += gap_minutes

            # Gap between last class and end
            last = day_sections[-1]
            penalty += int(day_end - last.time_end) // 60

    return penalty


def room_size_penalty(chromosome: list[SectionSchedule]) -> float:
    penalty = 0

    for schedule in chromosome:
        penalty += schedule.room.max_size - schedule.students_size # rooms are validated at runtime

    return penalty
````

## print_utils.py

````python
from functools import reduce

from rich.table import Table
from rich.console import Console
import copy

from genetic_algorithm.work.class_scheduling.domain import SectionSchedule, Time, Group, Professor, MeetingType

console = Console()


def format_time_no_seconds(time: Time):
    hours = time.hour
    minutes = time.minute

    return f"{hours:02d}:{minutes:02d}"

def section_type(meeting_type: MeetingType):
    if meeting_type == MeetingType.LAB:
        return "LABORATOR"

    elif meeting_type == MeetingType.COURSE:
        return "CURS"

    elif meeting_type == MeetingType.SEMINAR:
        return "SEMINAR"

    else:
        return ""

def display_week_schedule(chromosome: list[SectionSchedule], start_hour: Time, end_hour: Time, course_duration: Time, pause_duration: Time,
                          *, for_group=None, for_professor=None, for_room=None):
        days = ["Luni", "Marți", "Miercuri", "Joi", "Vineri"]
        hours = []

        st = copy.copy(start_hour)
        while st + course_duration <= end_hour:
            hours.append(f'{format_time_no_seconds(st)}-{format_time_no_seconds(st + course_duration)}')
            st += course_duration + pause_duration

        schedule = {day: {slot: "" for slot in hours} for day in days}

        for s in chromosome:
            if for_group and not any(g.name == for_group for g in s.section.group):
                continue

            if for_professor and not s.section.professor.name == for_professor:
                continue

            if for_room and not s.room.name == for_room:
                continue


            start = format_time_no_seconds(s.time_start)
            end = format_time_no_seconds(s.time_end)

            slot = f'{start}-{end}'
            day = days[s.day]

            if day in schedule and slot in schedule[day]:
                info = f"{s.section.subject.name}"

                if for_group:
                    info += "\n\n" + s.section.professor.name + "\n" + s.room.name

                if for_professor:
                    info += "\n"
                    info += "\n".join(g.name for g in s.section.group)
                    info += "\n\n" + s.room.name

                if for_room:
                    info += "\n"
                    info += "\n".join(g.name for g in s.section.group)
                    info += "\n\n" + s.section.professor.name

                info += "\n\n" + f"{section_type(s.section.meeting_type)}"
                schedule[day][slot] = info


        # Build the table
        title = f"Orar"
        if for_group:
            title += f" grupa {for_group}"

        elif for_professor:
            title += f" profesor {for_professor}"

        else:
            title += f" sală {for_room}"


        table = Table(title=title, show_lines=True, expand=True, min_width=300)

        table.add_column('Oră', justify='center', max_width=20, no_wrap=False)
        for d in days:
            table.add_column(d, justify='center', max_width=25)

        for slot in hours:
            row = [slot]
            for d in days:
                row.append(schedule[d][slot])
            table.add_row(*row)


        console.print(table)


def interactive_schedule(chromosome: list[SectionSchedule], start_hour, end_hour, course_duration, pause_duration, groups, professors,
                         rooms):

    group_names = [g.name for g in groups]
    professor_names = [p.name for p in professors]

    room_names = [r.name for r in rooms]

    while True:
        console.print('\n[bold]Selectează vizualizare:[/bold]')
        console.print("1. Grupă")
        console.print("2. Profesor")
        console.print("3. Sală")
        console.print("4. Ieșire")

        choice = input("Introdu alegerea ta: ").strip()

        if choice == '1':
            console.print(f"Grupe existente: \n{'\n'.join(group_names)}")
            console.print("\n")
            g = input('Introdu nume grupa: ').strip()
            if g not in group_names:
                console.print(f"[red]Grupa {g} nu a fost găsită![/red].")
                continue
            display_week_schedule(
                chromosome,
                start_hour,
                end_hour,
                course_duration,
                pause_duration,
                for_group=g,
            )

        elif choice == '2':
            console.print(f"Profesori existenți: \n{'\n'.join(professor_names)}")
            p = input('Introdu nume profesor: ').strip()
            if p not in professor_names:
                console.print(f"[red]Profesorul {p} nu a fost găsit![/red].")
                continue

            display_week_schedule(
                chromosome,
                start_hour,
                end_hour,
                course_duration,
                pause_duration,
                for_professor=p
            )
        elif choice == '3':
            console.print(f"Sali existente: \n{'\n'.join(room_names)}")
            p = input("Introdu nume sală:")
            if p not in room_names:
                console.print(f"[red]Sala {p} nu a fost găsită![/red]")

            display_week_schedule(
                chromosome,
                start_hour,
                end_hour,
                course_duration,
                pause_duration,
                for_room=p
            )
        elif choice == '4':
            console.print("[bold green]Ieșire din vizualizator...[/bold green]")
            break

        else:
            console.print("[red]Alegere invalidă, reîncearcă.[/red]")
````

